# K_01d – Reale Grid-Topologie & Netzfluss-Animationen

**Grid-Arbitrage** · Echte Übertragungs-Netzstruktur (OSM/Overpass) für CH und weitere DACH/EU-Länder

**Gruppe:** SC26_Gruppe_2 | **Status:** Kür – erweiterte Infrastruktur-Analyse | **Abhängig von:** K_01c

---

> **Ziel:** Realistische Netzknoten (Substations) und Kanten (Hochspannungsleitungen) statt synthetischer
> Zonenzentroide. Animierter Energiefluss auf echten Leitungspfaden.
>
> **Länder-generisch:** Alle Datenquellen und Funktionen sind über `CC_CODE` parametrisiert.
> Für CH ist ein vollständiger Offline-Fallback (27 Knoten, 34 Kanten, 380/220 kV) eingebaut.
>
> **Datenquellen (Priorität):**
> 1. **earth-osm** (Geofabrik PBF) — vollständigste OSM-Quelle, kein Timeout-Risiko
> 2. **Overpass API** (OSM) — direktes OSM, bei grossen Ländern ggf. Timeout
> 3. **PyPSA-Eur Zenodo** (zenodo.org/records/14144752) — preprocessed, mit elektr. Parametern
> 4. **Hardcoded Baseline** — CH-spezifisch, offline, aus Swissgrid-Publikationen
>
> ⚠️ Alle Lastfluss-Werte bleiben synthetisch modelliert.


| [← K_01c Animationen](K_01c_Energiefluss_Animationen.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_01d'></a>

1. [Initialisierung](#init_K_01d)
2. [Grid-Topologie laden](#topo_K_01d)
3. [Netzwerk-Graph aufbauen](#graph_K_01d)
4. [Statische Netzwerkkarte](#static_K_01d)
5. [Fluss-Modell auf Kanten](#flow_K_01d)
6. [GIF A — Tagesfluss auf Leitungen](#gif_a_K_01d)
7. [GIF B — Jahresverlauf & Richtungsumkehr](#gif_b_K_01d)
8. [Multi-Country Erweiterung](#multi_K_01d)
9. [Abschluss](#abschluss_K_01d)


---
## 1. Initialisierung<a id='init_K_01d'></a>

[↑ TOC](#toc_K_01d)

In [ ]:
# ── Bibliotheken ─────────────────────────────────────────────────────────────
import os, warnings, json, time
import subprocess, sys
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.lines import Line2D
import requests
warnings.filterwarnings('ignore')

for pkg, imp in [('geopandas','geopandas'),('networkx','networkx'),('pillow','PIL')]:
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

import geopandas as gpd
import networkx as nx

# ── Logging-Level: INFO→WARNING damit stderr sauber bleibt ─────────────
import logging
for _log in ['earth_osm','eo','eo.stream','eo.backends',
             'matplotlib','matplotlib.animation','PIL']:
    logging.getLogger(_log).setLevel(logging.WARNING)
# matplotlib animation INFO-Meldungen zusätzlich über rcParams unterdrücken
import matplotlib as _mpl
_mpl.rcParams['animation.convert_path'] = 'convert'

print(f"GeoPandas  : {gpd.__version__}")
print(f"NetworkX   : {nx.__version__}")
print(f"📅 Ausgeführt: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


In [ ]:
# ── Config (SSOT) + Länder-Parameter ─────────────────────────────────────────
with open('../sync/config.json') as _f: CFG = json.load(_f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']
CHARTS_DIR = os.path.join('../output', 'charts', SZ_AKTIV)
DATA_DIR   = os.path.join('../data', 'raw')
CACHE_DIR  = os.path.join('../data', 'intermediate', 'grid_topology')
os.makedirs(CHARTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,  exist_ok=True)
# EXP_: Experimental outputs → eigener Unterordner + EXP_-Präfix
EXP_CHARTS_DIR = os.path.join('../output', 'charts', 'experimental')
os.makedirs(EXP_CHARTS_DIR, exist_ok=True)


DPI_GIF  = 130
FPS_TAG  = 6
FPS_JAHR = 8

# ── ⚙ Länder-Konfiguration ──────────────────────────────────────────────────
# ISO 3166-1 Alpha-2 Codes — erweiterbar auf DE, AT, FR, IT, etc.
# ⚙ config.json → kuer.k01d.laender
COUNTRY_CONFIG = {
    'CH': {
        'name':        'Schweiz',
        'osm_area':    'ISO3166-1"="CH',
        'bbox':        (5.88, 45.78, 10.60, 47.92),   # (minlon, minlat, maxlon, maxlat)
        'voltage_kv':  [220, 380],                     # zu ladende Spannungsebenen
        'zone_mapping': {                               # Kanton → Zone (aus K_01b)
            'ZH':'Nordost','TG':'Nordost','SH':'Nordost','SG':'Nordost',
            'AR':'Ostschweiz','AI':'Ostschweiz','GR':'Ostschweiz',
            'GL':'Ostschweiz','SZ':'Ostschweiz',
            'AG':'Mitte-Erzeugung','SO':'Mitte-Erzeugung','BL':'Mitte-Erzeugung',
            'BE':'Mitte-Transit','LU':'Mitte-Transit','BS':'Mitte-Transit',
            'ZG':'Mitte-Transit','NW':'Mitte-Transit','OW':'Mitte-Transit',
            'VD':'West','GE':'West','NE':'West','JU':'West','FR':'West',
            'VS':'Süd','TI':'Süd','UR':'Süd',
        },
        'zone_colors': {
            'Nordost':'#1565C0','Ostschweiz':'#00838F',
            'Mitte-Erzeugung':'#7B1FA2','Mitte-Transit':'#388E3C',
            'West':'#FF6F00','Süd':'#B71C1C',
        },
        'map_xlim':    (5.88, 10.65),
        'map_ylim':    (45.78, 47.95),
    },
    'DE': {
        'name':        'Deutschland',
        'osm_area':    'ISO3166-1"="DE',
        'bbox':        (6.00, 47.20, 15.10, 55.10),
        'voltage_kv':  [220, 380],
        'zone_mapping': {},
        'zone_colors': {},
        'map_xlim':    (6.0, 15.2),
        'map_ylim':    (47.2, 55.2),
    },
    'AT': {
        'name':        'Österreich',
        'osm_area':    'ISO3166-1"="AT',
        'bbox':        (9.50, 46.35, 17.20, 48.80),
        'voltage_kv':  [220, 380],
        'zone_mapping': {},
        'zone_colors': {},
        'map_xlim':    (9.5, 17.2),
        'map_ylim':    (46.3, 48.9),
    },
}

# ⚙ Aktives Land — hier wechseln für andere Länder
CC_CODE = 'CH'  # ⚙ config.json → kuer.k01d.aktives_land
CC = COUNTRY_CONFIG[CC_CODE]

# Farben aus config
_viz         = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK      = _viz.get('bg_dark',    '#0d1117')
BG_PANEL     = _viz.get('bg_panel',   '#141414')
C_LOAD       = _viz.get('c_load',     '#66BB6A')
C_CHARGE     = _viz.get('c_charge',   '#1565C0')
C_FEED       = _viz.get('c_feed',     '#B71C1C')
C_PRICE      = _viz.get('c_price',    '#FFA726')
C_SPINE      = _viz.get('c_spine',    '#333333')
C_ACHSE      = _viz.get('c_achse',    '#aaaaaa')
C_TICK       = _viz.get('c_tick',     '#bbbbbb')
_stil        = CFG.get('visualisierung', {}).get('stil', {})
FS_TITEL     = _stil.get('schriftgroesse_titel', 13)
FS_KLEIN     = _stil.get('schriftgroesse_klein',  7)

# Leitungsfarben nach Spannung
VOLTAGE_COLORS = {380: '#EF5350', 220: '#FFA726', 150: '#42A5F5', 110: '#66BB6A'}
VOLTAGE_LW     = {380: 2.5,       220: 1.5,       150: 1.0,       110: 0.7}

print(f"Aktives Land: {CC_CODE} — {CC['name']}")
print(f"Spannungsebenen: {CC['voltage_kv']} kV")
print(f"Cache: {CACHE_DIR}")


---
## 2. Grid-Topologie laden<a id='topo_K_01d'></a>

[↑ TOC](#toc_K_01d)

Drei Quellen in Prioritätsreihenfolge: Overpass API → PyPSA-Eur Zenodo → Hardcoded Baseline.

### ⚙ Datenqualität & Granularität — auf einen Blick

| Element | Status | Quelle |
|---------|--------|--------|
| Knoten-Koordinaten | 📡 **Echt** | OSM (earth-osm/Overpass) oder Swissgrid-Baseline |
| Leitungspfade | 📡 **Echt** (wenn OSM) | OSM `power=line`, 220/380 kV |
| Lastfluss-Richtung/-Intensität | 🔬 **Modelliert** | Imbalance-Differenz, kein DC-Solver |
| Cross-Border [MW] | 🔬 **Modelliert** | Anteilsschlüssel DE/IT/FR/AT |

**Für echte Lastflüsse:** PyPSA mit Leitungsimpedanzen aus Zenodo-Dataset (enthält r, x).

#### Räumliche Granularität

| Ebene | CH | Übertragungs-Äquivalent |
|-------|----|------------------------|
| Kantone (aktuell K_01) | 26 | ~grobe Zonenstruktur |
| Bezirke | ~150 | mittelgross |
| Gemeinden | ~2'100 | feingranular, aber kein Lastfluss-Relevanz |
| **Substations (K_01d)** | **147 (Swissgrid)** | **✅ physikalisch korrekt** |

→ Substations sind die einzig sinnvolle Granularität für Netzfluss-Analyse.
Gemeinden wären für Verbrauchsverteilung interessant, nicht für Topologie.


In [ ]:
# ── Hardcoded Baseline CH (Swissgrid-Publikationen + ENTSO-E) ────────────────
# Fallback wenn alle Online-Quellen nicht verfügbar sind.
# Quellen: Swissgrid Strategisches Netz 2040, Fugro 3D-Netzvermessung, ENTSO-E Map
# ⚙ Für andere Länder: eigene Baseline in COUNTRY_CONFIG eintragen

CH_NODES_BASELINE = {
    # Knoten: lon, lat, voltage_kv, zone
    'Laufenburg':  (7.906, 47.567, 380, 'Mitte-Erzeugung'),
    'Beznau':      (8.231, 47.554, 380, 'Mitte-Erzeugung'),
    'Leibstadt':   (8.179, 47.597, 380, 'Mitte-Erzeugung'),
    'Gösgen':      (7.970, 47.367, 380, 'Mitte-Erzeugung'),
    'Rupperswil':  (8.093, 47.408, 380, 'Mitte-Erzeugung'),
    'Lachmatt':    (7.641, 47.515, 380, 'Mitte-Erzeugung'),
    'Laufenburg2': (7.914, 47.567, 220, 'Mitte-Erzeugung'),
    'Oftringen':   (7.924, 47.320, 220, 'Mitte-Erzeugung'),
    'Mettlen':     (8.569, 47.222, 380, 'Mitte-Transit'),
    'Mühleberg':   (7.269, 46.978, 380, 'Mitte-Transit'),
    'Bickigen':    (7.641, 46.782, 380, 'Mitte-Transit'),
    'Bassecourt':  (7.243, 47.352, 220, 'Mitte-Transit'),
    'Weiningen':   (8.449, 47.434, 380, 'Nordost'),
    'Samstagern':  (8.726, 47.213, 380, 'Nordost'),
    'Obfelden':    (8.450, 47.305, 380, 'Nordost'),
    'Breite':      (8.645, 47.508, 220, 'Nordost'),
    'Romanel':     (6.610, 46.577, 220, 'West'),
    'St-Triphon':  (6.882, 46.376, 220, 'West'),
    'Pradella':    (10.143,46.839, 220, 'Ostschweiz'),
    'Sils':        (9.775, 46.434, 220, 'Ostschweiz'),
    'Airolo':      (8.622, 46.531, 220, 'Süd'),
    'Göschenen':   (8.588, 46.669, 220, 'Süd'),
    'Mörel':       (8.061, 46.360, 380, 'Süd'),
    'Chippis':     (7.530, 46.290, 220, 'Süd'),
    'Riddes':      (7.234, 46.174, 220, 'Süd'),
    'Castione':    (9.037, 46.313, 380, 'Süd'),
    'Magadino':    (8.889, 46.168, 220, 'Süd'),
}

CH_EDGES_BASELINE = [
    # (von, nach, kV) — Swissgrid Hauptkorridore
    ('Laufenburg','Beznau',380), ('Beznau','Leibstadt',380), ('Leibstadt','Laufenburg',380),
    ('Beznau','Gösgen',380), ('Gösgen','Rupperswil',380), ('Rupperswil','Mettlen',380),
    ('Mettlen','Weiningen',380), ('Weiningen','Samstagern',380),
    ('Samstagern','Obfelden',380), ('Obfelden','Mettlen',380),
    ('Laufenburg','Lachmatt',380), ('Lachmatt','Rupperswil',380),
    ('Mühleberg','Bickigen',380), ('Bickigen','Mettlen',380),
    ('Mörel','Mettlen',380), ('Castione','Airolo',380),
    ('Bassecourt','Bickigen',220), ('Bassecourt','Mühleberg',220),
    ('Mühleberg','Romanel',220), ('Romanel','St-Triphon',220),
    ('St-Triphon','Riddes',220), ('Riddes','Chippis',220),
    ('Chippis','Mörel',220), ('Riddes','Castione',220),
    ('Göschenen','Airolo',220), ('Mettlen','Göschenen',220),
    ('Gösgen','Oftringen',220), ('Airolo','Magadino',220),
    ('Magadino','Castione',220), ('Castione','Pradella',220),
    ('Sils','Pradella',220), ('Breite','Weiningen',220),
    ('Breite','Samstagern',220), ('Weiningen','Obfelden',380),
]

COUNTRY_BASELINES = {
    'CH': {'nodes': CH_NODES_BASELINE, 'edges': CH_EDGES_BASELINE},
}
print(f"CH Baseline: {len(CH_NODES_BASELINE)} Knoten, {len(CH_EDGES_BASELINE)} Kanten")


In [ ]:
# ── Overpass-Downloader (generisch) ──────────────────────────────────────────
OVERPASS_MIRRORS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://overpass.openstreetmap.fr/api/interpreter',
    'https://maps.mail.ru/osm/tools/overpass/api/interpreter',
]

def overpass_query(q, timeout=60):
    '''OSM Overpass-Query gegen mehrere Mirror-Server mit Fallback.'''
    for mirror in OVERPASS_MIRRORS:
        try:
            r = requests.post(mirror, data=q, timeout=timeout)
            if r.status_code == 200:
                return r.json()
        except Exception as e:
            print(f'  Mirror {mirror.split("/")[2]}: {e}')
    return None

def download_grid_osm(cc_code, cc_cfg, cache_dir, force=False):
    '''
    Lädt Substations und Leitungen für ein Land via Overpass API.
    Cacht Ergebnisse als JSON. Gibt (nodes_df, edges_df) zurück oder None bei Fehler.
    '''
    nodes_file = os.path.join(cache_dir, f'{cc_code}_substations.json')
    edges_file = os.path.join(cache_dir, f'{cc_code}_lines.json')

    if not force and os.path.exists(nodes_file) and os.path.exists(edges_file):
        print(f'{cc_code}: Cache vorhanden — lade aus {cache_dir}')
        nodes_data = json.load(open(nodes_file))
        edges_data = json.load(open(edges_file))
        return _parse_osm(nodes_data, edges_data, cc_cfg)

    volt_filter = '|'.join(str(v*1000) for v in cc_cfg['voltage_kv'])
    area_tag    = cc_cfg['osm_area']
    bb          = cc_cfg['bbox']  # minlon, minlat, maxlon, maxlat
    bbox_str    = f"{bb[1]},{bb[0]},{bb[3]},{bb[2]}"  # Overpass: minlat,minlon,maxlat,maxlon

    print(f'{cc_code}: Lade Substations von Overpass...')
    q_nodes = f'''
[out:json][timeout:90][bbox:{bbox_str}];
(
  node["power"="substation"]["voltage"~"^({volt_filter})$"];
  way["power"="substation"]["voltage"~"^({volt_filter})$"];
);
out center;
'''
    nodes_data = overpass_query(q_nodes, timeout=95)
    if nodes_data is None:
        print(f'⚠️  Overpass Substations fehlgeschlagen für {cc_code}')
        return None

    print(f'{cc_code}: Lade Leitungen von Overpass...')
    q_edges = f'''
[out:json][timeout:90][bbox:{bbox_str}];
way["power"="line"]["voltage"~"^({volt_filter})$"];
out geom;
'''
    edges_data = overpass_query(q_edges, timeout=95)
    if edges_data is None:
        print(f'⚠️  Overpass Leitungen fehlgeschlagen für {cc_code}')
        return None

    # Cache
    json.dump(nodes_data, open(nodes_file,'w'), ensure_ascii=False)
    json.dump(edges_data, open(edges_file,'w'), ensure_ascii=False)
    print(f'{cc_code}: Gespeichert ({len(nodes_data.get("elements",[]))} Substations,'
          f' {len(edges_data.get("elements",[]))} Leitungen)')
    return _parse_osm(nodes_data, edges_data, cc_cfg)


def _parse_osm(nodes_data, edges_data, cc_cfg):
    '''Parst Overpass-JSON → DataFrames.'''
    # Substations
    rows = []
    for el in nodes_data.get('elements', []):
        tags = el.get('tags', {})
        volt = int(tags.get('voltage', 0)) // 1000
        if volt not in cc_cfg['voltage_kv']: continue
        if el['type'] == 'node':
            lon, lat = el['lon'], el['lat']
        else:
            c = el.get('center', {}); lon, lat = c.get('lon'), c.get('lat')
        if lon is None: continue
        rows.append({'id': str(el['id']), 'lon': lon, 'lat': lat,
                     'voltage_kv': volt,
                     'name': tags.get('name', tags.get('ref', str(el['id']))),
                     'operator': tags.get('operator', ''),
                     'zone': ''})
    df_nodes = pd.DataFrame(rows)

    # Lines (als Start+End-Punkt)
    erows = []
    for el in edges_data.get('elements', []):
        tags = el.get('tags', {})
        volt = int(tags.get('voltage', 0)) // 1000
        if volt not in cc_cfg['voltage_kv']: continue
        geom = el.get('geometry', [])
        if len(geom) < 2: continue
        erows.append({'id': str(el['id']),
                      'from_lon': geom[0]['lon'],  'from_lat': geom[0]['lat'],
                      'to_lon':   geom[-1]['lon'], 'to_lat':   geom[-1]['lat'],
                      'geom':     [(p['lon'], p['lat']) for p in geom],
                      'voltage_kv': volt,
                      'name': tags.get('name', tags.get('ref', ''))})
    df_edges = pd.DataFrame(erows)
    print(f'Parsed: {len(df_nodes)} Substations, {len(df_edges)} Leitungen')
    return df_nodes, df_edges


def load_pypsa_zenodo(cc_code, cache_dir, force=False):
    '''
    Lädt PyPSA-Eur prebuilt network von Zenodo (buses.csv + lines.csv).
    Filtert auf Land cc_code. Gibt (df_nodes, df_edges) zurück.
    Quelle: zenodo.org/records/14144752
    '''
    buses_cache = os.path.join(cache_dir, f'{cc_code}_pypsa_buses.csv')
    lines_cache = os.path.join(cache_dir, f'{cc_code}_pypsa_lines.csv')

    if not force and os.path.exists(buses_cache) and os.path.exists(lines_cache):
        df_nodes = pd.read_csv(buses_cache)
        df_edges = pd.read_csv(lines_cache)
        print(f'{cc_code}: PyPSA Cache geladen ({len(df_nodes)} Knoten, {len(df_edges)} Kanten)')
        return df_nodes, df_edges

    ZENODO_ID = '14144752'
    base_url  = f'https://zenodo.org/records/{ZENODO_ID}/files'
    try:
        print(f'{cc_code}: Lade PyPSA-Eur buses.csv von Zenodo...')
        rb = requests.get(f'{base_url}/buses.csv?download=1', timeout=60)
        rl = requests.get(f'{base_url}/lines.csv?download=1', timeout=60)
        if rb.status_code != 200 or rl.status_code != 200:
            print(f'⚠️  Zenodo: HTTP {rb.status_code} / {rl.status_code}')
            return None
        from io import StringIO
        df_buses = pd.read_csv(StringIO(rb.text))
        df_lines = pd.read_csv(StringIO(rl.text))
        # Filter auf Land
        df_b = df_buses[df_buses['country'] == cc_code].copy()
        df_l = df_lines[df_lines['bus0'].isin(df_b.index) | df_lines['bus1'].isin(df_b.index)].copy()
        df_b = df_b.rename(columns={'x':'lon','y':'lat','v_nom':'voltage_kv'})
        df_b['id']   = df_b.index.astype(str)
        df_b['zone'] = ''
        df_l['id']   = df_l.index.astype(str)
        df_b[['id','lon','lat','voltage_kv','zone']].to_csv(buses_cache, index=False)
        df_l.to_csv(lines_cache, index=False)
        print(f'{cc_code}: PyPSA Zenodo: {len(df_b)} Knoten, {len(df_l)} Kanten')
        return df_b[['id','lon','lat','voltage_kv','zone']], df_l
    except Exception as e:
        print(f'⚠️  PyPSA Zenodo Fehler: {e}')
        return None


def load_from_baseline(cc_code, cc_cfg):
    '''Baut DataFrames aus hardcodierten Baseline-Daten.'''
    baseline = COUNTRY_BASELINES.get(cc_code)
    if baseline is None:
        print(f'⚠️  Keine Baseline für {cc_code}')
        return None, None
    nodes = baseline['nodes']
    edges = baseline['edges']
    df_n = pd.DataFrame([
        {'id': name, 'lon': lon, 'lat': lat, 'voltage_kv': kv,
         'name': name, 'zone': zone}
        for name, (lon, lat, kv, zone) in nodes.items()
    ])
    # Nur Kanten mit bekannten Knoten
    valid = set(nodes.keys())
    df_e = pd.DataFrame([
        {'id': f'{n1}-{n2}', 'from_node': n1, 'to_node': n2, 'voltage_kv': kv,
         'from_lon': nodes[n1][0], 'from_lat': nodes[n1][1],
         'to_lon':   nodes[n2][0], 'to_lat':   nodes[n2][1]}
        for n1, n2, kv in edges if n1 in valid and n2 in valid
    ])
    print(f'{cc_code} Baseline: {len(df_n)} Knoten, {len(df_e)} Kanten')
    return df_n, df_e


def load_earth_osm(cc_code, cc_cfg, cache_dir, force=False,
                   voltage_min=110):
    '''
    Lädt Substations und Leitungen via earth-osm (Geofabrik PBF-Extrakt).
    Vollständigste OSM-Quelle: robuster als Overpass, kein Timeout-Risiko.
    Quelle: github.com/pypsa-meets-earth/earth-osm
    Lizenz: ODbL (OpenStreetMap)

    cc_code : ISO 3166-1 Alpha-2 (z.B. "CH", "DE", "AT")
    voltage_min : Minimale Spannung [kV] — filtert auf Hochspannungsnetz
    '''
    # earth-osm-Regionsnamen (Geofabrik-Extrakte)
    GEOFABRIK_NAMES = {
        'CH': 'switzerland', 'DE': 'germany', 'AT': 'austria',
        'FR': 'france', 'IT': 'italy', 'NL': 'netherlands',
        'BE': 'belgium', 'PL': 'poland', 'CZ': 'czech-republic',
        'ES': 'spain', 'PT': 'portugal', 'SE': 'sweden',
        'NO': 'norway', 'DK': 'denmark', 'GB': 'great-britain',
    }
    region = GEOFABRIK_NAMES.get(cc_code)
    if region is None:
        print(f'⚠️  Kein Geofabrik-Extrakt für {cc_code} konfiguriert')
        return None

    sub_cache   = os.path.join(cache_dir, f'{cc_code}_earthosm_substations.geojson')
    line_cache  = os.path.join(cache_dir, f'{cc_code}_earthosm_lines.geojson')
    earth_data  = os.path.join(cache_dir, 'earth_osm_raw')

    if not force and os.path.exists(sub_cache) and os.path.exists(line_cache):
        print(f'{cc_code}: earth-osm Cache geladen')
        try:
            gdf_sub  = gpd.read_file(sub_cache)
            gdf_line = gpd.read_file(line_cache)
            return _parse_earthosm(gdf_sub, gdf_line, cc_cfg, voltage_min)
        except Exception as e:
            print(f'  Cache-Fehler: {e}')

    # earth-osm installieren falls nötig
    try:
        from earth_osm.eo import save_osm_data as _eo_save
        # Unterdrücke verbose INFO-Logs (erscheinen sonst als roter stderr-Output)
        import logging as _lg
        for _n in ['earth_osm','eo','eo.stream','eo.backends']:
            _lg.getLogger(_n).setLevel(_lg.WARNING)
    except ImportError:
        print('earth-osm nicht installiert — installiere...')
        subprocess.check_call([sys.executable,'-m','pip','install',
                               'earth-osm','--break-system-packages','-q'])
        from earth_osm.eo import save_osm_data as _eo_save
        # Unterdrücke verbose INFO-Logs (erscheinen sonst als roter stderr-Output)
        import logging as _lg
        for _n in ['earth_osm','eo','eo.stream','eo.backends']:
            _lg.getLogger(_n).setLevel(_lg.WARNING)

    print(f'{cc_code}: Lade via earth-osm (Geofabrik: {region})...')
    print('  Kann 1–5 Min dauern beim ersten Download (~100–600 MB PBF).')
    try:
        os.makedirs(earth_data, exist_ok=True)
        _eo_save(
            region_list=[region],
            primary_name='power',
            feature_list=['substation', 'line'],
            update=False,
            mp=False,
            out_dir=earth_data,
            data_dir=earth_data,
            out_format='csv',
            out_aggregate=False,
            data_source='geofabrik',
        )
        # earth-osm Output: <data_dir>/out/<CC>_raw_substations.geojson
        cc_up = cc_code.upper()
        region_cap = region.replace('-','_').upper()[:2]  # DE, AT, CH, ...
        # Suche Output-Dateien (Naming variiert je Version)
        candidates_sub  = [
            os.path.join(earth_data, 'out', f'{cc_up}_raw_substations.geojson'),
            os.path.join(earth_data, 'out', f'{region}_raw_substations.geojson'),
        ]
        candidates_line = [
            os.path.join(earth_data, 'out', f'{cc_up}_raw_lines.geojson'),
            os.path.join(earth_data, 'out', f'{region}_raw_lines.geojson'),
        ]
        # Flexibler Pfad-Finder
        import glob
        sub_found  = next((p for p in candidates_sub  if os.path.exists(p)), None)
        line_found = next((p for p in candidates_line if os.path.exists(p)), None)
        if sub_found is None:
            hits = glob.glob(os.path.join(earth_data, '**', '*substation*'), recursive=True)
            sub_found = hits[0] if hits else None
        if line_found is None:
            hits = glob.glob(os.path.join(earth_data, '**', '*line*'), recursive=True)
            line_found = hits[0] if hits else None

        if sub_found is None or line_found is None:
            print(f'⚠️  earth-osm Output nicht gefunden für {cc_code}')
            return None

        # earth-osm CSV → GeoDataFrame
        import pandas as _pd_eo
        def _csv_to_gdf(path):
            df = _pd_eo.read_csv(path)
            if 'geometry' in df.columns:
                from shapely import wkt as _wkt
                df['geometry'] = df['geometry'].apply(
                    lambda g: _wkt.loads(g) if isinstance(g,str) else None)
                return gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
            # Fallback: lon/lat columns
            if 'lon' in df.columns and 'lat' in df.columns:
                return gpd.GeoDataFrame(df,
                    geometry=gpd.points_from_xy(df['lon'], df['lat']),
                    crs='EPSG:4326')
            return gpd.GeoDataFrame(df)
        gdf_sub  = _csv_to_gdf(sub_found)
        gdf_line = _csv_to_gdf(line_found)
        # Cache
        gdf_sub.to_file(sub_cache, driver='GeoJSON')
        gdf_line.to_file(line_cache, driver='GeoJSON')
        print(f'  {cc_code}: {len(gdf_sub)} Substations, {len(gdf_line)} Leitungen (raw)')
        return _parse_earthosm(gdf_sub, gdf_line, cc_cfg, voltage_min)

    except Exception as e:
        print(f'⚠️  earth-osm Fehler für {cc_code}: {e}')
        return None


def _parse_earthosm(gdf_sub, gdf_line, cc_cfg, voltage_min=110):
    '''Parst earth-osm GeoDataFrames → (df_nodes, df_edges).'''
    if gdf_sub.crs and gdf_sub.crs.to_epsg() != 4326:
        gdf_sub = gdf_sub.to_crs(epsg=4326)
    if gdf_line.crs and gdf_line.crs.to_epsg() != 4326:
        gdf_line = gdf_line.to_crs(epsg=4326)

    # Spannung aus 'voltage' Tag (Wert z.B. '220000' oder '220')
    def parse_volt(v):
        try:
            v = str(v).split(';')[0].strip()
            val = int(float(v))
            return val // 1000 if val > 1000 else val
        except: return 0

    # Substations → Knoten
    rows = []
    for _, row in gdf_sub.iterrows():
        volt = parse_volt(row.get('voltage', 0))
        if volt < voltage_min: continue
        if row.geometry is None: continue
        c = row.geometry.centroid
        rows.append({'id': str(row.get('id', row.name)),
                     'lon': c.x, 'lat': c.y,
                     'voltage_kv': volt,
                     'name': str(row.get('name', row.get('ref', ''))),
                     'zone': ''})
    df_nodes = pd.DataFrame(rows).drop_duplicates(subset='id')

    # Lines → Kanten (Mittelpunkt-Koordinaten)
    erows = []
    for _, row in gdf_line.iterrows():
        volt = parse_volt(row.get('voltage', 0))
        if volt < voltage_min: continue
        if row.geometry is None: continue
        coords = list(row.geometry.coords) if hasattr(row.geometry, 'coords') else []
        if len(coords) < 2: continue
        erows.append({'id': str(row.get('id', row.name)),
                      'from_lon': coords[0][0],  'from_lat': coords[0][1],
                      'to_lon':   coords[-1][0], 'to_lat':   coords[-1][1],
                      'voltage_kv': volt})
    df_edges = pd.DataFrame(erows)

    print(f'earth-osm parsed: {len(df_nodes)} Substations ≥{voltage_min}kV, ' +
          f'{len(df_edges)} Leitungen')
    return df_nodes, df_edges


In [ ]:
# ── Topologie laden (Prioritätskette) ────────────────────────────────────────
# Priorität 1:  earth-osm      — Geofabrik PBF, vollständigste OSM-Quelle
# Priorität 2:  Overpass API   — direktes OSM, kann bei grossen Ländern timeouten
# Priorität 3:  PyPSA-Eur Zenodo — preprocessed, elektrische Parameter vorhanden
# Priorität 4:  Hardcoded Baseline — offline, CH-spezifisch

FORCE_RELOAD = CFG.get('force_reload', {}).get('grid_topology', False)

df_nodes = None
df_edges = None
DATA_SOURCE = 'unbekannt'

print("=== Lade Grid-Topologie ===")

# Priorität 1: earth-osm (Geofabrik)
print("P1: earth-osm (Geofabrik)...")
result = load_earth_osm(CC_CODE, CC, CACHE_DIR, force=FORCE_RELOAD)
if result is not None and len(result[0]) > 0:
    df_nodes, df_edges = result
    DATA_SOURCE = 'earth-osm (Geofabrik/OSM)'

# Priorität 2: Overpass API
if df_nodes is None or len(df_nodes) == 0:
    print("P2: Overpass API...")
    result = download_grid_osm(CC_CODE, CC, CACHE_DIR, force=FORCE_RELOAD)
    if result is not None and len(result[0]) > 0:
        df_nodes, df_edges = result
        DATA_SOURCE = 'Overpass API (OSM)'

# Priorität 3: PyPSA-Eur Zenodo
if df_nodes is None or len(df_nodes) == 0:
    print("P3: PyPSA-Eur Zenodo...")
    result = load_pypsa_zenodo(CC_CODE, CACHE_DIR, force=FORCE_RELOAD)
    if result is not None and len(result[0]) > 0:
        df_nodes, df_edges = result
        DATA_SOURCE = 'PyPSA-Eur (Zenodo)'

# Priorität 4: Hardcoded Baseline
if df_nodes is None or len(df_nodes) == 0:
    print(f"P4: Hardcoded Baseline ({CC_CODE})...")
    df_nodes, df_edges = load_from_baseline(CC_CODE, CC)
    DATA_SOURCE = f'Baseline (hardcoded {CC_CODE})'

print(f"\n✅ Quelle: {DATA_SOURCE}")
print(f"   Knoten: {len(df_nodes)} Substations")
print(f"   Kanten: {len(df_edges)} Leitungen")
if df_nodes is not None and len(df_nodes) > 0:
    print(f"   Spannungen: {sorted(df_nodes['voltage_kv'].unique())} kV")


---
## 3. Netzwerk-Graph aufbauen<a id='graph_K_01d'></a>

[↑ TOC](#toc_K_01d)

NetworkX-Graph aus Substations (Knoten) und Leitungen (Kanten). Knotengrad = Anzahl Verbindungen → Proxy für lokale Netzwichtigkeit.

In [ ]:
# ── NetworkX-Graph aufbauen — mit OSM-Datenbereinigung ────────────────────────
import networkx as nx
import numpy as np

# CH Bounding Box (strict) — filtert grenznahe ausländische Nodes
BB = {'minlon': 5.88, 'maxlon': 10.65, 'minlat': 45.78, 'maxlat': 47.95}

def in_bbox(lon, lat, bb=BB, margin=0.10):
    return (bb['minlon']-margin <= lon <= bb['maxlon']+margin and
            bb['minlat']-margin <= lat <= bb['maxlat']+margin)

def clean_name(row_id, row_name):
    '''Gibt lesbaren Namen zurück: OSM-IDs werden durch Koordinaten-Ref ersetzt.'''
    n = str(row_name).strip() if row_name else ''
    if not n or n == 'nan' or n.lstrip('-').isdigit():
        return None  # wird später durch kV-Label ersetzt
    return n[:20]

# Nodes filtern + bauen
G = nx.Graph()

df_n_clean = df_nodes.copy()
# Strict bbox filter (Nodes ausserhalb CH entfernen)
mask = df_n_clean.apply(
    lambda r: in_bbox(float(r['lon']), float(r['lat'])), axis=1)
df_n_clean = df_n_clean[mask].copy()
print(f"Nodes nach bbox-Filter: {len(df_n_clean)} / {len(df_nodes)}")

# Deduplication: Nodes die < 800m auseinander liegen → zusammenführen
# (Overpass liefert oft mehrere Nodes pro physischer Substation)
from scipy.spatial import cKDTree
coords = df_n_clean[['lon','lat']].values.astype(float)
# 1 Grad ≈ 85 km in CH → 0.01° ≈ 850 m
MERGE_DEG = 0.008  # ~680 m
if len(coords) > 1:
    tree = cKDTree(coords)
    pairs = tree.query_pairs(MERGE_DEG)
    # Union-Find: Gruppen bilden
    parent = list(range(len(df_n_clean)))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b): parent[find(a)] = find(b)
    for a, b in pairs: union(a, b)
    # Repräsentant pro Gruppe: höchste Spannung + erster Eintrag
    groups = {}
    for idx in range(len(df_n_clean)):
        root = find(idx)
        groups.setdefault(root, []).append(idx)
    # Wähle Repräsentanten: max voltage, dann erste
    keep_rows = []
    for root, idxs in groups.items():
        sub = df_n_clean.iloc[idxs]
        best = sub.sort_values('voltage_kv', ascending=False).iloc[0]
        keep_rows.append(best)
    df_n_clean = pd.DataFrame(keep_rows).reset_index(drop=True)
    print(f"Nach Deduplication (≤{MERGE_DEG*111:.0f}km): {len(df_n_clean)} Nodes")

# Nodes zum Graph hinzufügen
for _, row in df_n_clean.iterrows():
    nid  = str(row['id'])
    name = clean_name(nid, row.get('name', ''))
    kv   = int(row.get('voltage_kv', 220))
    G.add_node(nid,
               lon=float(row['lon']),
               lat=float(row['lat']),
               voltage_kv=kv,
               zone=str(row.get('zone', '')),
               name=name if name else f'{kv}kV',
               display_name=name if name else None)

# Kanten: Snap Leitungs-Endpunkte auf nächste Substation ────────────────────
# Nur Kanten innerhalb CH, maximale Snap-Distanz 25 km
MAX_SNAP_DEG = 0.25  # ~21 km
node_ids   = list(G.nodes())
node_lons  = np.array([G.nodes[n]['lon'] for n in node_ids])
node_lats  = np.array([G.nodes[n]['lat'] for n in node_ids])
node_tree  = cKDTree(np.column_stack([node_lons, node_lats]))

added_edges = set()
skipped = 0

for _, row in df_edges.iterrows():
    try:
        fl, flo = float(row['from_lat']), float(row['from_lon'])
        tl, tlo = float(row['to_lat']),  float(row['to_lon'])
    except (KeyError, ValueError):
        skipped += 1; continue

    # Beide Endpunkte müssen in CH bbox liegen
    if not in_bbox(flo, fl) or not in_bbox(tlo, tl):
        skipped += 1; continue

    # Snap auf nächste Substations
    fd, fi = node_tree.query([flo, fl])
    td, ti = node_tree.query([tlo, tl])

    if fd > MAX_SNAP_DEG or td > MAX_SNAP_DEG:
        skipped += 1; continue

    n1, n2 = node_ids[fi], node_ids[ti]
    if n1 == n2: continue

    key = (min(n1,n2), max(n1,n2))
    kv  = int(row.get('voltage_kv', 220))
    if key not in added_edges:
        G.add_edge(n1, n2, voltage_kv=kv)
        added_edges.add(key)

print(f"Kanten: {G.number_of_edges()} hinzugefügt, {skipped} übersprungen")

# Isolierte Nodes entfernen (Overpass liefert oft Single-Substations ohne Leitungen)
isolated = list(nx.isolates(G))
G.remove_nodes_from(isolated)
print(f"Isolierte Nodes entfernt: {len(isolated)}")

# Knotengrad
degree = dict(G.degree())
nx.set_node_attributes(G, degree, 'degree')

major    = [n for n,d in degree.items() if d >= 3]
minor    = [n for n,d in degree.items() if d == 2]
isolated_r = [n for n,d in degree.items() if d <= 1]

print(f"\nGraph final: {G.number_of_nodes()} Knoten, {G.number_of_edges()} Kanten")
print(f"  Grosse Knoten (Grad ≥3): {len(major)}")
print(f"  Mittlere Knoten (Grad 2): {len(minor)}")
if major:
    top5 = sorted(degree.items(), key=lambda x: -x[1])[:5]
    print(f"  Top-5 nach Grad: {top5}")


---
## 4. Statische Netzwerkkarte<a id='static_K_01d'></a>

[↑ TOC](#toc_K_01d)

Leitungen nach Spannung (Farbe + Stärke), Knoten nach Grad (Grösse).

In [ ]:
# ── Kantonsgrenzen laden (aus K_01-Cache) ────────────────────────────────────
KANT_CANDIDATES = [
    os.path.join(DATA_DIR, 'kantone.gpkg'),
    os.path.join(DATA_DIR, 'swissboundaries3d.gpkg'),
]
KANT_NUM_TO_ABK = {
    1:'ZH',2:'BE',3:'LU',4:'UR',5:'SZ',6:'OW',7:'NW',8:'GL',
    9:'ZG',10:'FR',11:'SO',12:'BS',13:'BL',14:'SH',15:'AR',16:'AI',
    17:'SG',18:'GR',19:'AG',20:'TG',21:'TI',22:'VD',23:'VS',24:'NE',25:'GE',26:'JU',
}
gdf_kant = None
for path in KANT_CANDIDATES:
    if os.path.exists(path) and os.path.getsize(path) > 50_000:
        try:
            layers = gpd.list_layers(path)
            lname = next((l for l in layers['name'] if 'kanton' in l.lower()),
                         layers['name'].iloc[0])
            gdf_raw = gpd.read_file(path, layer=lname)
            if gdf_raw.crs and gdf_raw.crs.to_epsg() != 4326:
                gdf_raw = gdf_raw.to_crs(epsg=4326)
            kab_col = 'KAB' if 'KAB' in gdf_raw.columns else None
            if kab_col is None:
                for col in gdf_raw.columns:
                    s = gdf_raw[col].astype(str)
                    if s.str.isnumeric().all():
                        gdf_raw['KAB'] = s.astype(int).map(KANT_NUM_TO_ABK)
                        kab_col = 'KAB'; break
                    elif s.str.len().between(2,3).mean() > 0.8:
                        gdf_raw['KAB'] = s.str.upper().str[:2]
                        kab_col = 'KAB'; break
            gdf_kant = gdf_raw.copy()
            print(f"Kantone: {len(gdf_kant)} | {os.path.basename(path)}")
            break
        except Exception as e:
            print(f"  {e}")
if gdf_kant is None:
    print("⚠️  Kantone nicht gefunden — Karten ohne Grenzen")


In [ ]:
print("\n▶ Starte Erstellung: Statische Netzwerkkarte (1/3)")
# 📡 ECHTE TOPOLOGIE (earth-osm/Overpass/Zenodo/Baseline) | 🔬 SYNTHETISCHE Zonenfarben
# ── Statische Netzwerkkarte ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor('#090d14')
ax.set_xlim(*CC['map_xlim']); ax.set_ylim(*CC['map_ylim'])
ax.set_axis_off()

# Kantonsgrenzen
if gdf_kant is not None:
    gdf_kant.boundary.plot(ax=ax, color='#1e2d40', linewidth=0.7, zorder=1)
    try:
        gpd.GeoDataFrame(geometry=[gdf_kant.unary_union.boundary], crs='EPSG:4326').plot(
            ax=ax, color='#2a3d55', linewidth=2.0, zorder=2)
    except Exception:
        pass

# Kanten nach Spannung (dicker = höher)
for n1, n2, data in G.edges(data=True):
    kv  = data.get('voltage_kv', 220)
    col = VOLTAGE_COLORS.get(kv, '#888')
    lw  = VOLTAGE_LW.get(kv, 1.0)
    x1, y1 = G.nodes[n1]['lon'], G.nodes[n1]['lat']
    x2, y2 = G.nodes[n2]['lon'], G.nodes[n2]['lat']
    ax.plot([x1,x2],[y1,y2], color=col, lw=lw, alpha=0.75, zorder=3, solid_capstyle='round')

# Knoten: Grösse = Grad, Farbe = Zone oder Spannung
zone_col = CC.get('zone_colors', {})
for n, data in G.nodes(data=True):
    deg  = data.get('degree', 1)
    kv   = data.get('voltage_kv', 220)
    zone = data.get('zone', '')
    col  = zone_col.get(zone, VOLTAGE_COLORS.get(kv, '#ccc'))
    size = max(25, min(400, deg**2 * 15))
    ax.scatter(data['lon'], data['lat'], s=size, c=col,
               alpha=0.92, zorder=10, linewidths=0.5,
               edgecolors='white' if deg >= 3 else 'none')
    # Labels für grosse Knoten
    if deg >= 3:
        name = data.get('display_name') or data.get('name', '')
        if not name: continue
        name = name[:14]
        ax.text(data['lon']+0.04, data['lat']+0.03, name,
                color='white', fontsize=5.5, zorder=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', fc='#090d14', alpha=0.7, lw=0))

# Legende
volt_handles = [Line2D([0],[0], color=VOLTAGE_COLORS[kv], lw=VOLTAGE_LW[kv]+0.5,
                       label=f'{kv} kV') for kv in [380,220]]
size_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor='#aaa',
                       markersize=ms, label=f'Grad {d}')
                for ms, d in [(4,1),(7,3),(10,5)]]
ax.legend(handles=volt_handles+size_handles, loc='lower left',
          fontsize=7, framealpha=0.6, facecolor='#111', labelcolor='white')

ax.set_title(f'Übertragungnetz {CC["name"]} — {CC_CODE} | Quelle: {DATA_SOURCE}',
             color='white', fontsize=FS_TITEL, fontweight='bold')
fig.text(0.98, 0.02, '⚠️ Topologie: OSM/Baseline | Lastfluss: synthetisch',
         ha='right', va='bottom', color='#ff5252', fontsize=7, transform=fig.transFigure)

plt.tight_layout()
p = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{CC_CODE.lower()}_netz_statisch.png')
plt.savefig(p, dpi=120, bbox_inches='tight', facecolor=BG_DARK)
print(f"✅ Gespeichert: {p}")


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
from IPython.display import Image, display
import os
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01d_ch_netz_statisch.png')
if os.path.exists(_p):
    display(Image(filename=_p))
else:
    print(f'⚠️  Datei nicht gefunden: {_p}')


**Statische Netzwerkkarte CH**

Übertragungnetz CH. Leitungen: 380 kV (rot/dick), 220 kV (orange/dünn).
Knoten-Grösse = Netzgrad (wichtige Hubs: Beznau, Mettlen, Laufenburg).
Quelle: Echte OSM-Topologie wenn online, Swissgrid-Baseline (27 Knoten) wenn offline.

---
## 5. Fluss-Modell auf Kanten<a id='flow_K_01d'></a>

[↑ TOC](#toc_K_01d)

Synthetisches Lastfluss-Modell: Imbalance pro Zone → Fluss auf kürzestem Pfad im Graph.
Richtung und Intensität variieren saisonal und stündlich — **Richtungsumkehr ist modelliert**.

In [ ]:
# ── Zonen-Imbalance Zeitreihen (aus K_01c-Logik) ─────────────────────────────
HOURS = np.arange(24)
WEEKS = np.arange(52)
SAISONS = ['Winter','Frühling','Sommer','Herbst']
MONAT_KURZ = ['Jan','Feb','Mrz','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez']

# Installierte Kapazitäten [MW] und Verbrauch aus K_01c-Modell
ZONE_PROD_INSTALLED = {
    'Nordost':         {'Solar':1800,'Wasserkraft': 500,'Kernkraft':   0},
    'Ostschweiz':      {'Solar': 600,'Wasserkraft':2800,'Kernkraft':   0},
    'Mitte-Erzeugung': {'Solar':1200,'Wasserkraft': 800,'Kernkraft':5000},
    'Mitte-Transit':   {'Solar':1400,'Wasserkraft':1200,'Kernkraft':   0},
    'West':            {'Solar':1000,'Wasserkraft': 900,'Kernkraft':   0},
    'Süd':             {'Solar': 800,'Wasserkraft':8500,'Kernkraft':   0},
}
CF = {'Solar':0.12,'Wasserkraft':0.38,'Kernkraft':0.80}
CF_SEASONAL = {
    'Solar':      {'Winter':0.30,'Frühling':0.75,'Sommer':1.00,'Herbst':0.50},
    'Wasserkraft':{'Winter':0.65,'Frühling':0.95,'Sommer':0.90,'Herbst':0.72},
    'Kernkraft':  {'Winter':1.00,'Frühling':0.98,'Sommer':0.92,'Herbst':0.98},
}
KANTON_POP = {
    'ZH':1593583,'BE':1065607,'LU':433654,'UR':37208,'SZ':165539,'OW':39028,
    'NW':43921,'GL':40590,'ZG':130426,'FR':340765,'SO':279375,'BS':183709,
    'BL':297025,'SH':86928,'AR':58050,'AI':16293,'SG':530468,'GR':202461,
    'AG':718084,'TG':295373,'TI':358353,'VD':826400,'VS':357043,'NE':177589,
    'GE':511655,'JU':73584,
}
KANTON_TO_ZONE_B = CC['zone_mapping']
zone_pop = {}
for k, z in KANTON_TO_ZONE_B.items():
    zone_pop[z] = zone_pop.get(z,0) + KANTON_POP.get(k,0)
SPEZ_KW = 0.76

ZONE_BASE_CONS = {z: pop * SPEZ_KW / 1000 for z,pop in zone_pop.items()}

def solar_h(h): return max(0.0, float(np.sin(np.pi*(h-6.0)/13.0)))
def hydro_h(h): return float(1.0 + 0.22*np.exp(-((h-10.5)**2)/14) + 0.18*np.exp(-((h-19.0)**2)/10))
def cons_h(h, sais):
    s = {'Winter':1.15,'Frühling':1.00,'Sommer':0.87,'Herbst':1.02}.get(sais,1.0)
    return s*(1.0 + 0.38*np.exp(-((h-8.5)**2)/4.5) + 0.44*np.exp(-((h-19.0)**2)/5.0) - 0.28*np.exp(-((h-4.0)**2)/3.0))

def solar_w(w): return 0.12 + 0.10*np.sin(2*np.pi*(w-12)/52)
def hydro_w(w): return 0.38 + 0.14*np.sin(2*np.pi*(w-15)/52)
def nuclear_w(w): return 0.79 - 0.05*np.sin(2*np.pi*(w-26)/52)
def cons_w(w): return 1.00 - 0.14*np.sin(2*np.pi*(w-12)/52)

def zone_imb_h(zone, h, sais):
    prod = sum(inst * CF[et] * CF_SEASONAL[et][sais]
               * (solar_h(h) if et=='Solar' else (hydro_h(h) if et=='Wasserkraft' else 1.0))
               for et, inst in ZONE_PROD_INSTALLED.get(zone,{}).items())
    cons = ZONE_BASE_CONS.get(zone, 500) * cons_h(h, sais)
    return prod - cons

def zone_imb_w(zone, w):
    prod = sum(inst * (solar_w(w) if et=='Solar' else (hydro_w(w) if et=='Wasserkraft' else nuclear_w(w)))
               for et, inst in ZONE_PROD_INSTALLED.get(zone,{}).items())
    cons = ZONE_BASE_CONS.get(zone, 500) * cons_w(w)
    return prod - cons

# Kantonsgrenze CH: Winter Nacht-Stunde importiert CH?
print("Imbalance-Test Nordost (= Verbrauchszentrum):")
for sais in SAISONS:
    imb_00 = zone_imb_h('Nordost', 0, sais)
    imb_12 = zone_imb_h('Nordost', 12, sais)
    print(f"  {sais}: 00h={imb_00:+.0f} MW, 12h={imb_12:+.0f} MW")

print("\nCH Gesamt-Imbalance (Export+):")
for sais in SAISONS:
    total = sum(zone_imb_h(z, 0, sais) for z in ZONE_PROD_INSTALLED)
    print(f"  {sais} 00h: CH gesamt {total:+.0f} MW")

print("\nJahresverlauf CH gesamt (Winter↔Sommer):")
for w in [0, 13, 26, 39]:
    total_w = sum(zone_imb_w(z, w) for z in ZONE_PROD_INSTALLED)
    mn = MONAT_KURZ[int(w/52*12)]
    print(f"  Woche {w+1:02d} ({mn}): {total_w:+.0f} MW")


In [ ]:
# ── Kantens-Fluss auf Graph (physikalisch vereinfacht) ────────────────────────
# Modell: Überschuss-Zone exportiert via kürzestem Graphpfad zur Defizit-Zone.
# Mehrere Pfade überlagern sich → Kanten-Fluss = Summe der Teilflüsse.
# Richtung kann wechseln (Sommer: CH exportiert nach IT, Winter: importiert aus DE)

def compute_edge_flows(G, zone_fn, cc_cfg):
    '''
    Berechnet synthetischen Fluss auf jeder Kante.
    zone_fn(zone_name) -> imbalance_mw
    Gibt dict {(n1,n2): mw_flow} zurück (positiv = n1→n2).
    '''
    # Zonen-Imbalance
    zone_imb = {}
    for zone in cc_cfg.get('zone_colors', {}).keys():
        zone_imb[zone] = zone_fn(zone)

    # Knoten-Imbalance: Zone-Imbalance gleichmässig auf Knoten der Zone verteilt
    node_zone = nx.get_node_attributes(G, 'zone')
    zone_node_count = {}
    for n, z in node_zone.items():
        zone_node_count[z] = zone_node_count.get(z, 0) + 1

    node_surplus = {}
    for n, z in node_zone.items():
        cnt = zone_node_count.get(z, 1)
        node_surplus[n] = zone_imb.get(z, 0) / max(cnt, 1)

    # Für jeden Paarung (Überschuss→Defizit): kürzester Pfad, Fluss anteilig
    edge_flow = {(min(u,v),max(u,v)): 0.0 for u,v in G.edges()}

    surplus_nodes = [(n, s) for n,s in node_surplus.items() if s > 50]
    deficit_nodes = [(n, s) for n,s in node_surplus.items() if s < -50]

    for s_node, s_mw in surplus_nodes:
        for d_node, d_mw in deficit_nodes:
            if s_node == d_node: continue
            try:
                path = nx.shortest_path(G, s_node, d_node)
            except nx.NetworkXNoPath:
                continue
            # Fluss-Anteil proportional zum kleineren der beiden Werte
            flow = min(abs(s_mw), abs(d_mw)) * 0.3  # Dämpfung
            for i in range(len(path)-1):
                u, v = path[i], path[i+1]
                key = (min(u,v), max(u,v))
                # Richtung: positive = u→v
                if u < v:
                    edge_flow[key] = edge_flow.get(key, 0) + flow
                else:
                    edge_flow[key] = edge_flow.get(key, 0) - flow

    return edge_flow


# Cross-Border-Flüsse (vereinfacht wie K_01c)
BORDER_POINTS = {'DE':(7.60,47.72),'AT':(9.55,47.48),'IT':(8.95,45.92),'FR':(6.10,46.22)}
BORDER_COLORS = {'DE':'#42A5F5','AT':'#EF5350','IT':'#66BB6A','FR':'#FFA726'}
BORDER_WEIGHT  = {'DE':0.35,'AT':0.15,'IT':0.30,'FR':0.20}
CH_BORDER_NODES = {'DE':'Laufenburg','AT':'Pradella','IT':'Castione','FR':'Romanel'}

def get_border_flows(total_imb_mw):
    return {nb: total_imb_mw * w for nb, w in BORDER_WEIGHT.items()}

# Test
print("Edge-Flow Test Winter 00h:")
ef = compute_edge_flows(G,
     lambda z: zone_imb_h(z, 0, 'Winter'), CC)
top_flows = sorted(ef.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
for edge, flow in top_flows:
    print(f"  {edge[0]} → {edge[1]}: {flow:+.0f} MW")


---
## 6. GIF A — Tagesfluss auf Leitungen<a id='gif_a_K_01d'></a>

[↑ TOC](#toc_K_01d)

Moving Dots auf echten Leitungspfaden. Dot-Dichte & Grösse = MW-Fluss. 4 GIFs (je Jahreszeit) × 24 Frames.

In [ ]:
# ── Helper: Karte + Netz zeichnen ────────────────────────────────────────────
MAP_XLIM = CC['map_xlim']
MAP_YLIM = CC['map_ylim']

def draw_grid_base(ax, edge_flows=None, alpha_lines=0.35, alpha_nodes=0.5):
    '''Zeichnet Basiskarte + Leitungen + Knoten. Wenn edge_flows: Intensität angepasst.'''
    ax.set_xlim(*MAP_XLIM); ax.set_ylim(*MAP_YLIM)
    ax.set_facecolor('#090d14'); ax.set_axis_off()
    if gdf_kant is not None:
        gdf_kant.boundary.plot(ax=ax, color='#1a2535', linewidth=0.5, zorder=1)

    # Leitungen
    for n1, n2, data in G.edges(data=True):
        kv  = data.get('voltage_kv', 220)
        col = VOLTAGE_COLORS.get(kv, '#888')
        lw  = VOLTAGE_LW.get(kv, 1.0)
        if edge_flows:
            key = (min(n1,n2), max(n1,n2))
            flow = abs(edge_flows.get(key, 0))
            lw = lw + min(3.0, flow/800)
        x1,y1 = G.nodes[n1]['lon'],G.nodes[n1]['lat']
        x2,y2 = G.nodes[n2]['lon'],G.nodes[n2]['lat']
        ax.plot([x1,x2],[y1,y2], color=col, lw=lw, alpha=alpha_lines, zorder=2)

    # Knoten
    zone_col = CC.get('zone_colors', {})
    for n, data in G.nodes(data=True):
        deg  = data.get('degree', 1)
        zone = data.get('zone', '')
        kv   = data.get('voltage_kv', 220)
        col  = zone_col.get(zone, VOLTAGE_COLORS.get(kv, '#ccc'))
        size = max(12, min(180, deg**2 * 10))
        ax.scatter(data['lon'], data['lat'], s=size, c=col, alpha=alpha_nodes,
                   zorder=8, linewidths=0.4, edgecolors='white' if deg>=3 else 'none')


def draw_flow_dots_on_edge(ax, n1, n2, mw_flow, frame, n_frames,
                           min_mw=80, max_dots=8, base_size=20):
    '''Moving dots entlang echter Kante n1→n2.'''
    if abs(mw_flow) < min_mw: return
    x1,y1 = G.nodes[n1]['lon'],G.nodes[n1]['lat']
    x2,y2 = G.nodes[n2]['lon'],G.nodes[n2]['lat']
    if mw_flow < 0:
        x1,y1,x2,y2 = x2,y2,x1,y1

    kv  = G.edges[n1,n2].get('voltage_kv', 220) if G.has_edge(n1,n2) else 220
    col = VOLTAGE_COLORS.get(kv, '#FFA726')
    n_dots   = min(max_dots, max(2, int(abs(mw_flow)/400)))
    dot_size = min(120, max(base_size, abs(mw_flow)/20))
    speed    = 0.6 + abs(mw_flow)/4000

    for i in range(n_dots):
        t = (frame * speed / n_frames + i/n_dots) % 1.0
        x = x1 + t*(x2-x1)
        y = y1 + t*(y2-y1)
        ax.scatter(x, y, s=dot_size, c=col, alpha=0.90, zorder=12, linewidths=0, rasterized=True)


def draw_border_flow(ax, total_imb, frame, n_frames):
    '''Cross-Border Dots von/zu Grenzknoten.'''
    bflows = get_border_flows(total_imb)
    for nb, mw in bflows.items():
        if abs(mw) < 50: continue
        border_lon, border_lat = BORDER_POINTS[nb]
        anchor = CH_BORDER_NODES.get(nb)
        if anchor and anchor in G:
            ax_lon, ax_lat = G.nodes[anchor]['lon'], G.nodes[anchor]['lat']
        else:
            ax_lon, ax_lat = 8.15, 46.90  # CH-Zentrum Fallback
        col = BORDER_COLORS[nb]
        # Dots Richtung: positiv = Export (CH→Grenze)
        if mw >= 0:
            fl, la, tl, ta = ax_lon, ax_lat, border_lon, border_lat
        else:
            fl, la, tl, ta = border_lon, border_lat, ax_lon, ax_lat
        n_dots = min(6, max(1, int(abs(mw)/600)))
        for i in range(n_dots):
            t = (frame*0.7/n_frames + i/n_dots) % 1.0
            x = fl + t*(tl-fl); y = la + t*(ta-la)
            ax.scatter(x, y, s=35, c=col, alpha=0.88, zorder=13, linewidths=0, rasterized=True)
        ax.plot([ax_lon,border_lon],[ax_lat,border_lat], color=col, lw=0.6, alpha=0.25, zorder=4)
        ax.text(border_lon, border_lat+0.10, f'{nb}\n{mw:+.0f}MW',
                ha='center', va='bottom', color=col, fontsize=5, fontweight='bold',
                zorder=15, bbox=dict(boxstyle='round,pad=0.1', fc='#090d14', alpha=0.75, lw=0))

print("✅ Helper-Funktionen geladen")


In [ ]:
print("\n▶ Starte Erstellung: Tagesfluss 4 Saisons (2/3)")
# 📡 ECHTE TOPOLOGIE | 🔬 SYNTHETISCHER Lastfluss (Imbalance-Modell)
# ── GIF A: Tagesfluss, 4 Jahreszeiten × 24 Frames ────────────────────────────
def make_gif_tag(saison, out_path):
    def update(frame, fig, ax):
        ax.clear()
        for txt in fig.texts: txt.set_visible(False)
        fig.texts.clear()
        h = frame

        ef = compute_edge_flows(G, lambda z: zone_imb_h(z, h, saison), CC)
        total_imb = sum(zone_imb_h(z, h, saison) for z in ZONE_PROD_INSTALLED)

        draw_grid_base(ax, edge_flows=ef, alpha_lines=0.25, alpha_nodes=0.45)

        # Dots auf Kanten
        for (n1, n2), mw in ef.items():
            draw_flow_dots_on_edge(ax, n1, n2, mw, frame, 24)

        # Cross-Border
        draw_border_flow(ax, total_imb, frame, 24)

        # Zone-Imbalance Labels
        zone_col = CC.get('zone_colors', {})
        for zone, col in zone_col.items():
            imb = zone_imb_h(zone, h, saison)
            # Zone-Zentroid aus Knotenmittelpunkt
            z_nodes = [n for n,d in G.nodes(data=True) if d.get('zone')==zone]
            if z_nodes:
                cx = np.mean([G.nodes[n]['lon'] for n in z_nodes])
                cy = np.mean([G.nodes[n]['lat'] for n in z_nodes])
                ax.text(cx, cy, f'{imb:+.0f}', ha='center', va='center',
                        color='white', fontsize=5.5, fontweight='bold', zorder=14,
                        bbox=dict(boxstyle='round,pad=0.12', fc=col, alpha=0.65, lw=0))

        ax.set_title(f'Lastfluss {CC["name"]} — {h:02d}:00 Uhr ({saison})',
                     color='white', fontsize=11, fontweight='bold')
        fig.text(0.02, 0.97, f'⚙ Modelliert | Quelle: {DATA_SOURCE}',
                 ha='left', va='top', color='#aaa', fontsize=7, transform=fig.transFigure)
        fig.text(0.98, 0.97, '⚠️ Synthetische Lastflüsse',
                 ha='right', va='top', color='#ff5252', fontsize=7, transform=fig.transFigure)

    fig = plt.figure(figsize=(14, 8), facecolor=BG_DARK)
    ax  = fig.add_subplot(111)
    anim = FuncAnimation(fig, lambda f: update(f, fig, ax),
                         frames=24, interval=int(1000/FPS_TAG), blit=False)
    anim.save(out_path, writer=PillowWriter(fps=FPS_TAG), dpi=DPI_GIF)
    plt.close(fig)
    kb = os.path.getsize(out_path)//1024
    print(f"✅ {os.path.basename(out_path)} ({kb} KB)")

for saison in SAISONS:
    p = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{CC_CODE.lower()}_tag_{saison.lower()}.gif')
    make_gif_tag(saison, p)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
from IPython.display import Image, display
import os
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01d_ch_tag_winter.gif')
if os.path.exists(_p):
    display(Image(filename=_p))
else:
    print(f'⚠️  Datei nicht gefunden: {_p}')


**GIF A — Tagesfluss auf echten Leitungspfaden (Winter)**

Moving Dots auf echten OSM-Leitungen. Dot-Dichte = MW-Fluss.
Zone-Labels ±MW. Cross-Border an echten Grenzknoten (Laufenburg→DE, Castione→IT).

---
## 7. GIF B — Jahresverlauf & Richtungsumkehr<a id='gif_b_K_01d'></a>

[↑ TOC](#toc_K_01d)

52 Wochen. Die CH-Bilanz kippt von Export (Sommer) zu Import (Winter-Spitze). Richtungsumkehr der Cross-Border-Dots sichtbar.

In [ ]:
print("\n▶ Starte Erstellung: Jahresverlauf 19h + 12h (3/3)")
# 📡 ECHTE TOPOLOGIE | 🔬 SYNTHETISCH | ✅ RICHTUNGSUMKEHR modelliert
# ── GIF B: Jahresverlauf, 52 Wochen ──────────────────────────────────────────
def make_gif_jahr(out_path, hour_of_day=19):
    '''Jahresverlauf bei fixer Tageszeit (default 19h = Abendspitze).'''
    def update(frame, fig, ax):
        ax.clear()
        for txt in fig.texts: txt.set_visible(False)
        fig.texts.clear()
        w = frame
        monat = MONAT_KURZ[int(w/52*12)]

        # Wöchentliche Imbalance (Saisonalität + Tageszeit-Profil bei hour_of_day)
        # Pseudo-Saison aus Wochennummer
        saison_idx = int((w / 52) * 4)
        saison = SAISONS[saison_idx % 4]
        ef = compute_edge_flows(G, lambda z: zone_imb_h(z, hour_of_day, saison)
                                           * float(cons_w(w) / 1.0), CC)
        total_imb = sum(zone_imb_h(z, hour_of_day, saison) * float(cons_w(w))
                        for z in ZONE_PROD_INSTALLED)

        draw_grid_base(ax, edge_flows=ef, alpha_lines=0.22, alpha_nodes=0.45)
        for (n1, n2), mw in ef.items():
            draw_flow_dots_on_edge(ax, n1, n2, mw, frame, 52)
        draw_border_flow(ax, total_imb, frame, 52)

        # CH-Gesamtsaldo sichtbar machen
        col_saldo = C_LOAD if total_imb > 0 else C_FEED
        saldo_txt = f'CH: {total_imb:+.0f} MW\n{"Export ↗" if total_imb>0 else "Import ↙"}'
        ax.text(8.35, 47.60, saldo_txt, ha='center', va='center',
                color='white', fontsize=7, fontweight='bold', zorder=16,
                bbox=dict(boxstyle='round,pad=0.25', fc=col_saldo, alpha=0.80, lw=0))

        ax.set_title(f'Jahresverlauf {CC["name"]} — Woche {w+1:02d}/52 ({monat}) | {hour_of_day:02d}:00 Uhr',
                     color='white', fontsize=11, fontweight='bold')
        fig.text(0.02, 0.97, f'⚙ Modelliert | {DATA_SOURCE}',
                 ha='left', va='top', color='#aaa', fontsize=7, transform=fig.transFigure)
        fig.text(0.98, 0.97, '⚠️ Synthetische Lastflüsse',
                 ha='right', va='top', color='#ff5252', fontsize=7, transform=fig.transFigure)

    fig = plt.figure(figsize=(14, 8), facecolor=BG_DARK)
    ax  = fig.add_subplot(111)
    anim = FuncAnimation(fig, lambda f: update(f, fig, ax),
                         frames=52, interval=int(1000/FPS_JAHR), blit=False)
    anim.save(out_path, writer=PillowWriter(fps=FPS_JAHR), dpi=DPI_GIF)
    plt.close(fig)
    kb = os.path.getsize(out_path)//1024
    print(f"✅ {os.path.basename(out_path)} ({kb} KB)")

# 19h-Variante (Abendspitze — hier ist CH im Winter Nettoimporteur)
p19 = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{CC_CODE.lower()}_jahr_19h.gif')
make_gif_jahr(p19, hour_of_day=19)

# 12h-Variante (Mittagsstunde — Solarproduktion sichtbar)
p12 = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{CC_CODE.lower()}_jahr_12h.gif')
make_gif_jahr(p12, hour_of_day=12)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
from IPython.display import Image, display
import os
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01d_ch_jahr_19h.gif')
if os.path.exists(_p):
    display(Image(filename=_p))
else:
    print(f'⚠️  Datei nicht gefunden: {_p}')


**GIF B — Jahresverlauf Abendspitze (19h)**

Richtungsumkehr: CH-Saldo-Badge wechselt grün (Export) ↔ rot (Import).
19h: Abendspitze → Importrisiko im Winter. 12h: Mittag → Solar-Überschuss Sommer.

---
## 8. Multi-Country Erweiterung<a id='multi_K_01d'></a>

[↑ TOC](#toc_K_01d)

Anleitung und Code-Template für DE, AT, FR, IT. Alle Funktionen sind generisch — nur `CC_CODE` ändern.

### 8.1 Workflow für ein neues Land

1. **`COUNTRY_CONFIG` ergänzen** (in Initialisierungszelle):
```python
'DE': {
    'name': 'Deutschland',
    'osm_area': 'ISO3166-1"="DE',
    'bbox': (6.00, 47.20, 15.10, 55.10),
    'voltage_kv': [220, 380],
    'zone_mapping': {},     # ggf. Bundesland-Zonen definieren
    'zone_colors': {},
    'map_xlim': (6.0, 15.2),
    'map_ylim': (47.2, 55.2),
}
```

2. **`CC_CODE = 'DE'`** setzen → Notebook komplett neu ausführen

3. Optional **Baseline** für DE in `COUNTRY_BASELINES` hinterlegen
   (TSO-Umspannwerke: Amprion, TenneT, TransnetBW, 50Hertz)

4. **GIFs** erscheinen automatisch mit `kuer_k01d_de_*` Prefix

### 8.2 Datenqualität nach Land

| Land | Overpass-Qualität | Baseline vorhanden | Hinweis |
|------|------------------|--------------------|---------|
| CH | ★★★★★ | ✅ 27 Knoten | Swissgrid gut dokumentiert |
| DE | ★★★★☆ | ❌ (noch offen) | 4 TSOs, > 200 Knoten |
| AT | ★★★★☆ | ❌ | APG (~60 Knoten) |
| FR | ★★★☆☆ | ❌ | RTE, teilweise fehlend in OSM |
| IT | ★★★☆☆ | ❌ | Terna, OSM unvollständig |

### 8.3 PyPSA-Eur als beste Vollösung

Für DE/AT/FR/IT empfehlen sich die **prebuilt networks** von Zenodo:
- `zenodo.org/records/14144752` — alle CH/DE/AT/FR/IT buses + lines als CSV
- Direkt filterbar nach `country` Spalte
- Reaktanz/Widerstand vorhanden (für echten DC-Lastfluss)


In [ ]:
# ── Multi-Country Batch (alle konfigurierten Länder) ─────────────────────────
# Aktivieren mit: BATCH_RUN = True
# Erzeugt statische Karten für alle Länder mit verfügbaren Daten

BATCH_RUN = False  # ⚙ True für automatischen Multi-Country-Durchlauf

if BATCH_RUN:
    for cc, cc_cfg in COUNTRY_CONFIG.items():
        print(f"\n{'='*50}")
        print(f"=== {cc}: {cc_cfg['name']} ===")

        # Topologie laden
        result = download_grid_osm(cc, cc_cfg, CACHE_DIR, force=False)
        if result is None:
            result = load_from_baseline(cc, cc_cfg)
        if result is None:
            print(f"⚠️  Keine Daten für {cc} — übersprungen")
            continue

        df_n, df_e = result

        # Graph aufbauen
        G_cc = nx.Graph()
        for _, row in df_n.iterrows():
            G_cc.add_node(row['id'], lon=float(row['lon']), lat=float(row['lat']),
                          voltage_kv=int(row.get('voltage_kv',220)),
                          zone=str(row.get('zone','')))
        for _, row in df_e.iterrows():
            fn = row.get('from_node', row.get('id',''))
            tn = row.get('to_node', row.get('id',''))
            if fn in G_cc and tn in G_cc and fn != tn:
                G_cc.add_edge(fn, tn, voltage_kv=int(row.get('voltage_kv',220)))
        nx.set_node_attributes(G_cc, dict(G_cc.degree()), 'degree')

        # Statische Karte
        fig, ax = plt.subplots(figsize=(14, 9))
        fig.patch.set_facecolor(BG_DARK)
        ax.set_facecolor('#090d14')
        ax.set_xlim(*cc_cfg['map_xlim']); ax.set_ylim(*cc_cfg['map_ylim'])
        ax.set_axis_off()
        for n1, n2, data in G_cc.edges(data=True):
            kv  = data.get('voltage_kv', 220)
            x1,y1 = G_cc.nodes[n1]['lon'],G_cc.nodes[n1]['lat']
            x2,y2 = G_cc.nodes[n2]['lon'],G_cc.nodes[n2]['lat']
            ax.plot([x1,x2],[y1,y2], color=VOLTAGE_COLORS.get(kv,'#888'),
                    lw=VOLTAGE_LW.get(kv,1), alpha=0.7, zorder=2)
        for n, d in G_cc.nodes(data=True):
            size = max(10, min(250, d.get('degree',1)**2 * 10))
            ax.scatter(d['lon'], d['lat'], s=size, c=VOLTAGE_COLORS.get(d.get('voltage_kv',220),'#ccc'),
                       alpha=0.85, zorder=8, linewidths=0)
        ax.set_title(f'Netz {cc_cfg["name"]} ({cc}) | {G_cc.number_of_nodes()} Knoten',
                     color='white', fontsize=13, fontweight='bold')
        plt.tight_layout()
        p = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{cc.lower()}_netz_statisch.png')
        plt.savefig(p, dpi=110, bbox_inches='tight', facecolor=BG_DARK)
        plt.close()
        print(f"✅ {p}")
else:
    print("BATCH_RUN = False — nur CH aktiv. Für alle Länder: BATCH_RUN = True setzen.")


---
## 9. Abschluss<a id='abschluss_K_01d'></a>

[↑ TOC](#toc_K_01d)

In [ ]:
# ── Abschlusskontrolle K_01d ─────────────────────────────────────────────────
print('K_01d – Abschlusskontrolle')
print('='*60)
cc = CC_CODE.lower()
expected = [
    f'EXP_kuer_k01d_{cc}_netz_statisch.png',
    f'EXP_kuer_k01d_{cc}_tag_winter.gif',
    f'EXP_kuer_k01d_{cc}_tag_frühling.gif',
    f'EXP_kuer_k01d_{cc}_tag_sommer.gif',
    f'EXP_kuer_k01d_{cc}_tag_herbst.gif',
    f'EXP_kuer_k01d_{cc}_jahr_19h.gif',
    f'EXP_kuer_k01d_{cc}_jahr_12h.gif',
]
total_kb = 0
ok = 0
for fname in expected:
    p = os.path.join(EXP_CHARTS_DIR, fname)
    exists = os.path.exists(p)
    kb = os.path.getsize(p)//1024 if exists else 0
    print(f"  {'✅' if exists else '❌'} {fname:<50} {kb:>5} KB")
    if exists: ok += 1; total_kb += kb

print()
print(f"  Total: {ok}/{len(expected)} | {total_kb//1024:.1f} MB")
print()
print(f"  Graph: {G.number_of_nodes()} Knoten, {G.number_of_edges()} Kanten")
print(f"  Quelle: {DATA_SOURCE}")
print()
print('  Multi-Country: BATCH_RUN = True → alle konfigurierten Länder')
print('  Neue Länder:  COUNTRY_CONFIG[\"XX\"] eintragen → CC_CODE = \"XX\"')


| [← K_01c Animationen](K_01c_Energiefluss_Animationen.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|